# 02 — Data Preprocessing

Test and visualize the preprocessing pipeline: resizing, normalization, augmentation, and data generators.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import tensorflow as tf

from src.data.preprocessing import DataPreprocessor
from src.data.augmentation import DataAugmentor
from src.data.dataset import AnimalDataset
from src.utils.config import load_config

%matplotlib inline
config = load_config('../config/config.yaml')
CLASS_NAMES = config['data']['classes']
print(f'Config loaded. Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

## 1. Run Preprocessing Pipeline

In [ ]:
preprocessor = DataPreprocessor(config_path='../config/config.yaml')

# Run preprocessing (set force=True to re-process from scratch)
preprocessor.run(force=False)

# Show statistics
stats = preprocessor.get_split_statistics()
print(f'\n{"Class":<12} {"Train":>8} {"Val":>8} {"Test":>8} {"Total":>8}')
print('-' * 50)
all_classes = set()
for s in stats.values():
    all_classes.update(s.keys())

for cls in sorted(all_classes):
    t  = stats.get('train', {}).get(cls, 0)
    v  = stats.get('validation', {}).get(cls, 0)
    te = stats.get('test', {}).get(cls, 0)
    print(f'{cls:<12} {t:>8} {v:>8} {te:>8} {t+v+te:>8}')

total_train = sum(stats.get('train', {}).values())
total_val   = sum(stats.get('validation', {}).values())
total_test  = sum(stats.get('test', {}).values())
print('-' * 50)
print(f'{"TOTAL":<12} {total_train:>8} {total_val:>8} {total_test:>8} {total_train+total_val+total_test:>8}')

## 2. Verify Processed Images (224×224)

In [ ]:
processed_dir = Path('../data/processed/train')

if processed_dir.exists():
    cols = 5
    rows = 3
    fig, axes = plt.subplots(rows, cols, figsize=(16, 10))
    axes = axes.flatten()

    for i, cls in enumerate(CLASS_NAMES):
        cls_dir = processed_dir / cls
        if cls_dir.exists():
            imgs = list(cls_dir.glob('*.jpg'))[:1] + list(cls_dir.glob('*.jpeg'))[:1]
            if imgs:
                img = Image.open(imgs[0])
                axes[i].imshow(img)
                axes[i].set_title(f'{cls}\n{img.size[0]}×{img.size[1]}', fontsize=9)
                axes[i].axis('off')

    plt.suptitle('Processed Images — All 15 Classes (224×224)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/figures/processed_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Processed data not found. Run preprocessing first.')

## 3. Data Augmentation Visualization

In [ ]:
processed_dir = Path('../data/processed/train')

if processed_dir.exists():
    sample_images = list(processed_dir.rglob('*.jpg'))[:1]
    if sample_images:
        img = Image.open(sample_images[0]).convert('RGB')
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_tensor = tf.expand_dims(img_array, 0)

        augmentor = DataAugmentor()
        aug_layers = augmentor.get_augmentation_layers()

        fig, axes = plt.subplots(2, 5, figsize=(16, 7))
        axes = axes.flatten()

        axes[0].imshow(img_array)
        axes[0].set_title('Original', fontweight='bold', color='blue')
        axes[0].axis('off')

        for i in range(1, 10):
            aug_img = aug_layers(img_tensor, training=True)[0].numpy()
            aug_img = np.clip(aug_img, 0, 1)
            axes[i].imshow(aug_img)
            axes[i].set_title(f'Augmented #{i}', fontsize=9)
            axes[i].axis('off')

        plt.suptitle('Data Augmentation Examples', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../outputs/figures/augmentation_examples.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('Processed data not found.')

## 4. Build Data Generators and Inspect a Batch

In [ ]:
dataset = AnimalDataset(config_path='../config/config.yaml')
train_gen, val_gen, test_gen = dataset.build_generators(augment_train=True)

print(f'Train samples:      {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Test samples:       {test_gen.samples}')
print(f'Batch size:         {train_gen.batch_size}')
print(f'Steps per epoch:    {len(train_gen)}')
print(f'\nClass indices:')
for cls, idx in sorted(train_gen.class_indices.items(), key=lambda x: x[1]):
    print(f'  {idx:>2}  {cls}')

In [ ]:
images, labels = next(iter(train_gen))
print(f'Batch images shape: {images.shape}')
print(f'Batch labels shape: {labels.shape}')
print(f'Pixel value range:  [{images.min():.3f}, {images.max():.3f}]')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for i in range(8):
    axes[i].imshow(images[i])
    cls_idx = np.argmax(labels[i])
    axes[i].set_title(CLASS_NAMES[cls_idx].capitalize(), fontsize=10)
    axes[i].axis('off')
plt.suptitle('Training Batch Sample', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Class Weights (for imbalance handling)

In [ ]:
class_weights = dataset.get_class_weights()
print('Class weights (higher = more weight given to underrepresented class):')
for idx, weight in sorted(class_weights.items()):
    print(f'  {idx:>2}  {CLASS_NAMES[idx]:<12}  {weight:.4f}')